<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/LIB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsed and Annotated Data

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 257 (delta 43), reused 4 (delta 4), pack-reused 175 (from 1)
Receiving objects: 100% (257/257), 15.04 MiB | 10.35 MiB/s, done.
Resolving deltas: 100% (94/94), done.


In [2]:
import pandas as pd
import io
import requests

box_files = {
    "PostMalone.csv":    "https://virginia.box.com/shared/static/o6v4vb3c3pe7kwysmbho7e8agi1co4z4.csv",
    "TaylorSwift.csv":   "https://virginia.box.com/shared/static/ynww2btxjuvl1cei9yaawxm0jgkrqvfs.csv",
    "EdSheeran.csv":     "https://virginia.box.com/shared/static/6zgbpge5yklaiiyypq62eif15axxrm07.csv",
    "JustinBieber.csv":  "https://virginia.box.com/shared/static/k92hby4d8pcbqc2ka9kpeicu4wv0jaiw.csv",
    "ColdPlay.csv":      "https://virginia.box.com/shared/static/2mb65mgblh008fg5s6sfio3bdey1eiph.csv",
    "Maroon5.csv":       "https://virginia.box.com/shared/static/v0az2kge2s8lycapyvzsmx813bb6avkt.csv",
    "Beyonce.csv":       "https://virginia.box.com/shared/static/3giiurqlx8jrs1t83gx0fisstxzv0wy8.csv",
    "LadyGaga.csv":      "https://virginia.box.com/shared/static/uzwtc77cxjauk2vn5xoeqms50prw5of9.csv",
    "BillieEilish.csv":  "https://virginia.box.com/shared/static/airnc5s6mw6c3zrieqps4e3nxc0stk0m.csv",
    "ArianaGrande.csv":  "https://virginia.box.com/shared/static/8m1ekqm8l303le447r3vzgpn773xo9pv.csv",
}

dfs = []
for filename, url in box_files.items():
    df = pd.read_csv(url)
    df['_source_file'] = filename
    dfs.append(df)
    print(f"Loaded {filename}")

source_data = pd.concat(dfs, ignore_index=True)
print(f"\nCorpus shape: {source_data.shape}")
source_data.head()


Loaded PostMalone.csv
Loaded TaylorSwift.csv
Loaded EdSheeran.csv
Loaded JustinBieber.csv
Loaded ColdPlay.csv
Loaded Maroon5.csv
Loaded Beyonce.csv
Loaded LadyGaga.csv
Loaded BillieEilish.csv
Loaded ArianaGrande.csv

Corpus shape: (3073, 8)


,Unnamed: 0,Artist,Title,Album,Year,Date,Lyric,_source_file
0,0.0,Post Malone,​​rockstar,beerbongs & bentleys,2017.0,2017-09-15,post malone hahahahaha tank god ayy ayy post...,PostMalone.csv
1,1.0,Post Malone,White Iverson,Stoney (Deluxe),2015.0,2015-02-04,double ot i'm a new three saucin' saucin' i'...,PostMalone.csv
2,2.0,Post Malone,Congratulations,Stoney (Deluxe),2016.0,2016-11-04,post malone mmmmm yeah yeah mmmmm yeah hey p...,PostMalone.csv
3,3.0,Post Malone,Psycho,beerbongs & bentleys,2018.0,2018-02-23,post malone damn my ap goin' psycho lil' mama ...,PostMalone.csv
4,4.0,Post Malone,I Fall Apart,Stoney (Deluxe),2016.0,2016-12-09,ooh i fall apart ooh yeah mmm yeah she told ...,PostMalone.csv


In [3]:
source_data = source_data[~source_data['Lyric'].str.contains('lyrics for this song have yet to be released please check back once the song has been released', case=False, na=False)]
print(source_data.shape)

(2964, 8)


## LIB

In [4]:
source_data['track_id'] = source_data['Artist'] + ' — ' + source_data['Title'] + ' (' + source_data['Album'] + ')'

# add additional columns for model summarization
source_data['doc_length_words'] = source_data['Lyric'].astype(str).str.split().str.len()
source_data['doc_length_chars'] = source_data['Lyric'].astype(str).str.len()
source_data['Year'] = pd.to_numeric(source_data['Year'], errors='coerce')
source_data['Decade'] = (source_data['Year'].dropna().astype(int) // 10 * 10).astype(str) + 's'

LIB = (source_data[['track_id', 'Artist', 'Album', 'Title', 'Year', 'Decade',
                     'doc_length_words', 'doc_length_chars']]
       .drop_duplicates()
       .set_index('track_id'))

print(f" LIB shape: {LIB.shape}")
LIB.head()

 LIB shape: (2964, 7)


,Artist,Album,Title,Year,Decade,doc_length_words,doc_length_chars
track_id,,,,,,,
Post Malone — ​​rockstar (beerbongs & bentleys),Post Malone,beerbongs & bentleys,​​rockstar,2017.0,2010s,507,2560
Post Malone — White Iverson (Stoney (Deluxe)),Post Malone,Stoney (Deluxe),White Iverson,2015.0,2010s,466,2324
Post Malone — Congratulations (Stoney (Deluxe)),Post Malone,Stoney (Deluxe),Congratulations,2016.0,2010s,533,2697
Post Malone — Psycho (beerbongs & bentleys),Post Malone,beerbongs & bentleys,Psycho,2018.0,2010s,536,2691
Post Malone — I Fall Apart (Stoney (Deluxe)),Post Malone,Stoney (Deluxe),I Fall Apart,2016.0,2010s,320,1552


In [5]:
LIB.to_csv('/content/DS5001-Final-Project/LIB.csv', sep='|', index=True)

In [6]:
print("Average length of document, in characters:", source_data['Lyric'].str.len().mean())

Average length of document, in characters: 1640.675280516831
